# AEGIS-MD — full pipeline in one notebook (GPU T4 + Internet ON)
Sections 1–5 = the five notebooks. Run cells in order; every section saves its outputs to /kaggle/working and the next section reloads them.


In [ ]:
%%writefile aegis_core.py
# =====================================================================
# aegis_core.py  —  AEGIS-MD core library
# Adaptive Evidence-Gap Investigation of Safety signals for Medical Devices
#
# Shared by all notebooks (each notebook writes this file to disk first,
# so every notebook is self-contained on Kaggle).
# =====================================================================
import os, re, glob, json, math, time, warnings
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from scipy import stats, sparse

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# Configuration (change here, or override in a notebook before use)
# ---------------------------------------------------------------------
CFG = dict(
    DATA_START="2014-01",   # first month of MAUDE data pulled
    DATA_END="2025-12",     # last month of MAUDE data pulled
    PANEL_START="2016-01",  # first evaluation month t
    PANEL_END="2025-06",    # last evaluation month t (needs HORIZON months of labels after it)
    TRAIN_END="2020-12",    # t <= TRAIN_END  -> train
    VAL_END="2022-06",      # TRAIN_END < t <= VAL_END -> validation ; later -> test
    HORIZON=6,              # predict a new recall initiated in (t, t+HORIZON]
    WASHOUT=6,              # drop t if a recall of that code was initiated in (t-WASHOUT, t]
    RECENT=3,               # recent window = months t-2..t
    BASELINE=12,            # baseline window = 12 months before the recent window
    LABEL_CLASSES=["Class I"],   # ["Class I"] (primary) or ["Class I","Class II"]
    K_PER_GAP=10,           # documents read per evidence-gap retrieval
    SEED=42,
)

DEEP_SOURCES = ["mechanism", "investigation", "batch", "precedent"]
TIER0_SOURCES = ["statistical", "temporal", "ml_anomaly", "severity", "problem_shift"]

# Gap specifications: what each evidence gap needs, and how it is queried.
GAP_SPECS = {
    "mechanism": dict(
        question="Is there a consistent, specific failure mechanism behind the recent reports?",
        proto="The device failed or malfunctioned in a specific, repeatable way; the report describes the failure mechanism and the component involved.",
        terms=["failure", "failed", "malfunction", "broke", "crack", "leak", "error", "alarm", "component"],
        use_emerging=True),
    "investigation": dict(
        question="What did the manufacturer's own investigation of returned devices conclude?",
        proto="The manufacturer evaluated the returned device; the investigation confirmed the reported failure and identified the root cause.",
        terms=["evaluation", "evaluated", "investigation", "returned", "analysis", "confirmed", "root", "cause", "found", "reproduced"],
        use_emerging=False),
    "batch": dict(
        question="Are the affected units concentrated in specific lots, batches or models?",
        proto="Multiple units from the same lot or manufacturing batch were affected by the same manufacturing defect.",
        terms=["lot", "batch", "manufacturing", "units", "same", "defect", "production", "nonconformance", "supplier"],
        use_emerging=False),
    "precedent": dict(
        question="Does this failure resemble defects that previously led to recalls?",
        proto="The failure resembles a known product defect that previously led to a corrective action or product recall.",
        terms=["defect", "design", "manufacturing", "failure", "component"],
        use_emerging=True),
}
GENERIC_QUERY = dict(
    proto="A report of an adverse event or problem involving the medical device.",
    terms=["adverse", "event", "problem", "malfunction", "patient", "device"])

# ---------------------------------------------------------------------
# Month helpers  (month index mi = year*12 + month-1)
# ---------------------------------------------------------------------
def ym(s):
    y, m = map(int, str(s)[:7].split("-")); return y * 12 + m - 1

def ym_str(mi):
    mi = int(mi); return f"{mi // 12}-{mi % 12 + 1:02d}"

def to_mi(series):
    d = pd.to_datetime(series, errors="coerce", format="mixed") if hasattr(pd, "to_datetime") else series
    return (d.dt.year * 12 + d.dt.month - 1).astype("float")

def mi_from_yyyymmdd(s):
    s = str(s or "")
    s = s.replace("-", "")
    if len(s) < 6 or not s[:6].isdigit():
        return np.nan
    return int(s[:4]) * 12 + int(s[4:6]) - 1

def split_of(mi, cfg=CFG):
    if mi <= ym(cfg["TRAIN_END"]): return "train"
    if mi <= ym(cfg["VAL_END"]): return "val"
    return "test"

# ---------------------------------------------------------------------
# File discovery (Kaggle: /kaggle/working and attached inputs)
# ---------------------------------------------------------------------
def data_roots():
    roots = []
    if os.environ.get("AEGIS_DIR"): roots.append(os.environ["AEGIS_DIR"])
    roots += ["/kaggle/working", "/kaggle/input", "."]
    return [r for r in roots if os.path.isdir(r)]

def find_file(name, required=True):
    for r in data_roots():
        p = os.path.join(r, name)
        if os.path.exists(p): return p
        hits = glob.glob(os.path.join(r, "**", name), recursive=True)
        if hits: return sorted(hits, key=len)[0]
    if required:
        raise FileNotFoundError(f"{name} not found. Attach the previous notebook's output as an input (Add Input -> Notebook Output).")
    return None

def out_dir():
    d = os.environ.get("AEGIS_OUT") or ("/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
    os.makedirs(d, exist_ok=True); return d

# ---------------------------------------------------------------------
# Text utilities
# ---------------------------------------------------------------------
STOP = set("""a an the and or of to in on for with without by from at as is are was were be been being this that these those it its
into than then there their they them he she his her patient patients device devices reported report reports reportedly per also
not no yes any all some such which who whom what when where while during after before about over under between via
had has have having did does do done will would could should may might can cannot unable however therefore further
information additional event events date dates time times use used using user customer facility hospital
mfr manufacturer company firm product products model lot serial number numbers unknown na n/a""".split())
TOKEN_RE = re.compile(r"[a-z][a-z0-9\-]{2,}")

def tokenize(s, keep_stop=False):
    toks = TOKEN_RE.findall(str(s or "").lower())
    return toks if keep_stop else [t for t in toks if t not in STOP]

# Leakage guard: MAUDE narratives can be updated by supplemental reports
# after a recall. Every sentence that mentions a recall / field action is removed.
RECALL_MENTION = re.compile(
    r"(recall|recalled|field safety|field action|field correction|correction and removal|"
    r"\bfsca\b|\bfsn\b|z-\d{3,4}-\d{4}|urgent medical device|advisory notice|removal notice)", re.I)
SENT_SPLIT = re.compile(r"(?<=[\.\!\?;])\s+")

def scrub_recall_mentions(text):
    if not text: return "", 0
    sents = SENT_SPLIT.split(str(text))
    keep = [s for s in sents if not RECALL_MENTION.search(s)]
    return " ".join(keep), len(sents) - len(keep)

# Regex stance (CPU fallback when no NLI model is available)
CONFIRM_RE = re.compile(
    r"(was|were|has been|have been|is) confirmed|confirmed the (reported|complaint|failure|issue|allegation)|"
    r"root cause (was|is|has been) (determined|identified|attributed|found)|"
    r"(failure|problem|issue|defect) (was|is|were) (found|observed|reproduced|duplicated|verified)|"
    r"(found|observed) to be (cracked|broken|defective|damaged|leaking|fractured|separated|occluded)|"
    r"(evaluation|analysis|investigation|inspection) (found|revealed|identified|determined|showed)", re.I)
REFUTE_RE = re.compile(
    r"could not be (confirmed|reproduced|duplicated|determined|verified)|"
    r"no (device )?(malfunction|anomal\w*|defect|problem|issue|fault|failure)s? (was|were )?(found|identified|observed|detected|noted)|"
    r"(was|were|has not been|have not been) not (been )?returned|not returned for (evaluation|analysis)|unable to (confirm|reproduce|duplicate|verify)|"
    r"functioned (as intended|properly|normally|as designed)|(met|meets|within) (all )?(specification|specifications)|"
    r"\b(user|use) error\b|not confirmed|unconfirmed|no evidence of (a )?(device )?(malfunction|defect)", re.I)

def regex_stance(text):
    t = str(text or "")
    if not t.strip(): return np.nan, np.nan
    c = 0.85 if CONFIRM_RE.search(t) else 0.10
    r = 0.85 if REFUTE_RE.search(t) else 0.10
    return c, r

# ---------------------------------------------------------------------
# Sparse BM25 index over all records (vectorised; fast per-case scoring)
# ---------------------------------------------------------------------
class BM25Matrix:
    def __init__(self, texts, k1=1.2, b=0.75, min_df=3, max_features=60000):
        from sklearn.feature_extraction.text import CountVectorizer
        self.cv = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None,
                                  min_df=min_df, max_features=max_features)
        tf = self.cv.fit_transform(texts).tocsr().astype(np.float32)
        N = tf.shape[0]
        df = np.diff(tf.tocsc().indptr)
        idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(tf.sum(1)).ravel()
        avgdl = max(dl.mean(), 1.0)
        tf = tf.tocoo()
        denom = tf.data + k1 * (1 - b + b * dl[tf.row] / avgdl)
        w = idf[tf.col] * tf.data * (k1 + 1) / denom
        self.W = sparse.csr_matrix((w, (tf.row, tf.col)), shape=tf.shape)
        self.vocab = self.cv.vocabulary_

    def qvec(self, tokens):
        idx = sorted({self.vocab[t] for t in tokens if t in self.vocab})
        v = np.zeros(self.W.shape[1], dtype=np.float32); v[idx] = 1.0
        return v

    def score(self, rows, qv):
        return np.asarray(self.W[rows] @ qv).ravel()

def _minmax(x):
    x = np.asarray(x, dtype=np.float32)
    lo, hi = (x.min(), x.max()) if len(x) else (0, 0)
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)

def hybrid_retrieve(rows, bm25, emb, q_tokens, proto_vec, k, alpha=0.5):
    """Hybrid sparse (BM25) + dense (embedding) retrieval restricted to candidate rows."""
    rows = np.asarray(rows)
    if len(rows) <= k: return rows
    s_bm = _minmax(bm25.score(rows, bm25.qvec(q_tokens)))
    s_dn = _minmax(emb[rows].astype(np.float32) @ proto_vec.astype(np.float32))
    s = alpha * s_bm + (1 - alpha) * s_dn
    return rows[np.argsort(-s, kind="stable")[:k]]

# ---------------------------------------------------------------------
# Tier-0 (routine surveillance) detectors
# ---------------------------------------------------------------------
def mann_kendall(x):
    x = np.asarray(x, dtype=float); n = len(x)
    if n < 4 or np.all(x == x[0]): return 0.0, 0.0, 1.0
    s = 0.0
    for i in range(n - 1): s += np.sign(x[i + 1:] - x[i]).sum()
    _, counts = np.unique(x, return_counts=True)
    var = (n * (n - 1) * (2 * n + 5) - np.sum(counts * (counts - 1) * (2 * counts + 5))) / 18.0
    z = 0.0 if var <= 0 else ((s - 1) / math.sqrt(var) if s > 0 else ((s + 1) / math.sqrt(var) if s < 0 else 0.0))
    tau = s / (0.5 * n * (n - 1))
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return tau, z, p

def sen_slope(x):
    x = np.asarray(x, dtype=float); n = len(x)
    if n < 3: return 0.0
    sl = [(x[j] - x[i]) / (j - i) for i in range(n - 1) for j in range(i + 1, n)]
    return float(np.median(sl))

def jsd(p_counts, q_counts):
    keys = set(p_counts) | set(q_counts)
    if not keys: return 0.0
    p = np.array([p_counts.get(k, 0) for k in keys], float) + 0.5
    q = np.array([q_counts.get(k, 0) for k in keys], float) + 0.5
    p /= p.sum(); q /= q.sum(); m = 0.5 * (p + q)
    return float(0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m)))

def tier0_for_code(n_by_month, months, agg, cfg=CFG):
    """Tier-0 evidence for every month of one product code.
    n_by_month: dict mi->total reports (openFDA counts); agg: dict mi->dict of sample aggregates."""
    R, B = cfg["RECENT"], cfg["BASELINE"]
    rows, cusum = [], 0.0
    for t in months:
        rec = [n_by_month.get(m, 0) for m in range(t - R + 1, t + 1)]
        base = [n_by_month.get(m, 0) for m in range(t - R - B + 1, t - R + 1)]
        mu = max(np.mean(base), 0.25)
        n_rec = sum(rec)
        p_pois = stats.poisson.sf(n_rec - 1, mu * R)
        last12 = [n_by_month.get(m, 0) for m in range(t - 11, t + 1)]
        tau, z_mk, p_mk = mann_kendall(last12)
        slope = sen_slope(last12) / (np.mean(last12) + 1.0)
        persist = sum(stats.poisson.sf(n_by_month.get(m, 0) - 1, mu) < 0.05 for m in range(t - 5, t + 1))
        n_t = n_by_month.get(t, 0)
        cusum = max(0.0, cusum + (n_t - mu) / math.sqrt(mu + 1.0) - 0.5)
        # sample-based structured aggregates (severity, problem-code mix, reporting reliability)
        def pool(ms):
            out = dict(n=0, serious=0, mfr_src=0, text=0, prob=Counter())
            for m in ms:
                a = agg.get(m)
                if a is None: continue
                out["n"] += a["n"]; out["serious"] += a["serious"]; out["mfr_src"] += a["mfr_src"]
                out["text"] += a["text"]; out["prob"].update(a["prob"])
            return out
        w = pool(range(t - R + 1, t + 1)); b = pool(range(t - R - B + 1, t - R + 1))
        p1 = (w["serious"] + 0.5) / (w["n"] + 1); p0 = (b["serious"] + 0.5) / (b["n"] + 1)
        pp = (w["serious"] + b["serious"] + 1) / (w["n"] + b["n"] + 2)
        se = math.sqrt(pp * (1 - pp) * (1 / (w["n"] + 1) + 1 / (b["n"] + 1)))
        rows.append(dict(
            mi=t, n_recent=n_rec, base_mu=mu,
            pois_score=min(-math.log10(max(p_pois, 1e-12)), 12.0),
            log_ratio=math.log((n_rec + 0.5) / (mu * R + 0.5)),
            mk_z=z_mk, mk_tau=tau, sen_norm=slope, persist=persist, cusum=cusum,
            sev_z=(p1 - p0) / se if se > 0 else 0.0, sev_share=p1,
            prob_jsd=jsd(w["prob"], b["prob"]),
            reliab=(w["mfr_src"] + 1) / (w["n"] + 2), completeness=(w["text"] + 1) / (w["n"] + 2),
            n_sample_recent=w["n"]))
    return pd.DataFrame(rows)

TIER0_GROUPS = {
    "statistical": ["pois_score", "log_ratio", "cusum"],
    "temporal": ["mk_z", "sen_norm", "persist"],
    "ml_anomaly": ["if_score"],
    "severity": ["sev_z", "sev_share"],
    "problem_shift": ["prob_jsd"],
}

def emerging_terms(win_prob, base_prob, topn=3):
    tw = sum(win_prob.values()) + 1; tb = sum(base_prob.values()) + 1
    sc = {k: math.log((v + 1) / tw) - math.log((base_prob.get(k, 0) + 1) / tb) for k, v in win_prob.items()}
    top = [k for k, _ in sorted(sc.items(), key=lambda kv: -kv[1])[:topn]]
    toks = []
    for k in top: toks += tokenize(k)
    return top, toks

# ---------------------------------------------------------------------
# Deep evidence extracted from a set of retrieved documents S
# ---------------------------------------------------------------------
DEEP_FEATURES = {
    "mechanism": ["mech_coh", "mech_shift"],
    "investigation": ["inv_confirm", "inv_refute", "inv_cov"],
    "batch": ["lot_hhi", "model_hhi", "lot_cov"],
    "precedent": ["prec_top", "prec_max"],
}

def _hhi(values):
    v = [x for x in values if x]
    if len(v) < 2: return 0.0
    c = np.array(list(Counter(v).values()), float); s = c / c.sum()
    return float((s ** 2).sum())

def deep_evidence(S, R, ctx, sources=DEEP_SOURCES):
    """S: array of record row-ids; R: dict of record arrays; ctx: case context (baseline centroid etc.).
    Returns dict feature->value plus per-source reliability rho_<source>."""
    out = {}
    S = np.asarray(S, dtype=int)
    if len(S) == 0:
        for s in sources:
            for f in DEEP_FEATURES[s]: out[f] = np.nan
            out["rho_" + s] = 0.0
        return out
    rho = 0.5 + 0.5 * float(np.mean(R["has_text"][S] & R["mand_src"][S]))
    if "mechanism" in sources:
        E = R["emb"][S].astype(np.float32)
        c = E.mean(0); cn = c / (np.linalg.norm(c) + 1e-9)
        out["mech_coh"] = float((E @ cn).mean()) - ctx["base_coh"]
        out["mech_shift"] = float(1 - cn @ ctx["base_cen"]) if ctx["base_cen"] is not None else 0.0
        out["rho_mechanism"] = rho
    if "investigation" in sources:
        has = R["has_mfr"][S]
        if has.any():
            out["inv_confirm"] = float(np.nanmean(R["p_confirm"][S][has]))
            out["inv_refute"] = float(np.nanmean(R["p_refute"][S][has]))
        else:
            out["inv_confirm"] = out["inv_refute"] = np.nan
        out["inv_cov"] = float(has.mean())
        out["rho_investigation"] = rho
    if "batch" in sources:
        lots = R["lot"][S]; models = R["model"][S]
        out["lot_hhi"] = _hhi(lots); out["model_hhi"] = _hhi(models)
        out["lot_cov"] = float(np.mean([bool(x) for x in lots]))
        out["rho_batch"] = rho
    if "precedent" in sources:
        ps = np.sort(R["prec_sim"][S])[::-1]
        ps = ps[np.isfinite(ps)]
        out["prec_top"] = float(ps[:3].mean()) if len(ps) else np.nan
        out["prec_max"] = float(ps[0]) if len(ps) else np.nan
        out["rho_precedent"] = rho
    return out

LEVEL_FEATURES = {"mech_coh", "inv_confirm", "inv_refute", "lot_hhi", "model_hhi", "prec_top", "prec_max"}

def contrastive_gap_evidence(S_w, S_b, R, ctx, source, rate=None):
    """Query-conditioned contrastive evidence: the same gap query is run on the recent window
    (S_w) and on the code's own baseline window (S_b); level features are reported as
    window-minus-baseline differences so that query-induced selection bias cancels out."""
    fw = deep_evidence(S_w, R, ctx, sources=[source])
    fb = deep_evidence(S_b, R, ctx, sources=[source]) if len(S_b) else None
    out = {}
    for f in DEEP_FEATURES[source]:
        v = fw.get(f, np.nan)
        if f in LEVEL_FEATURES and fb is not None and np.isfinite(fb.get(f, np.nan)) and np.isfinite(v):
            v = v - fb[f]
        out[f] = v
    if source == "mechanism" and len(S_w) and len(S_b):
        cw = R["emb"][S_w].astype(np.float32).mean(0); cb = R["emb"][S_b].astype(np.float32).mean(0)
        out["mech_shift"] = float(1 - (cw @ cb) / (np.linalg.norm(cw) * np.linalg.norm(cb) + 1e-9))
    out["rate_" + source] = rate if rate is not None else np.nan
    out["rho_" + source] = fw.get("rho_" + source, 0.0)
    return out

def query_rate_shift(win, base, emb, proto, thr):
    """Change in the share of reports that match a gap query (dense similarity above a fixed
    corpus-level threshold) between the recent and the baseline window. No reading needed."""
    if len(win) == 0: return np.nan
    hw = float((emb[win].astype(np.float32) @ proto >= thr).sum())
    hb = float((emb[base].astype(np.float32) @ proto >= thr).sum()) if len(base) else 0.0
    return math.log((hw + 0.5) / (len(win) + 1)) - math.log((hb + 0.5) / (len(base) + 1))

CONTRAST_FEATURES = {s: DEEP_FEATURES[s] + ["rate_" + s] for s in DEEP_FEATURES}

# ---------------------------------------------------------------------
# Reliability-discounted Bayesian evidence fusion + Value-of-Information
# ---------------------------------------------------------------------
def _logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def _sigmoid(z): return 1 / (1 + np.exp(-z))

class SourceCalibrator:
    """Maps one evidence source's raw features to a log-likelihood ratio (LLR)
    relative to the base rate: LLR = logit(P(y|x)) - logit(a)."""
    def __init__(self, feats, C=0.5):
        self.feats, self.C = feats, C
    def _prep(self, X):
        X = X[self.feats].astype(float).copy()
        miss = X.isna().all(axis=1).values
        X = X.fillna(self.med)
        return X.values, miss
    def fit(self, X, y):
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        self.med = X[self.feats].median()
        self.med = self.med.fillna(0.0)
        Xv, miss = self._prep(X)
        self.sc = StandardScaler().fit(Xv[~miss])
        self.lr = LogisticRegression(C=self.C, max_iter=2000).fit(self.sc.transform(Xv[~miss]), y[~miss])
        self.la = _logit(np.clip(y.mean(), 1e-4, 1 - 1e-4))
        return self
    def llr(self, X):
        Xv, miss = self._prep(X)
        p = self.lr.predict_proba(self.sc.transform(Xv))[:, 1]
        l = _logit(p) - self.la
        l[miss] = 0.0          # absent evidence carries no information
        return np.clip(l, -6.0, 6.0)   # bound any single source's influence

class EvidenceFusion:
    """z = b0 + sum_k w_k * rho_k * LLR_k   (reliability-discounted log-odds pooling).
    Calibrators are fit on TRAIN; pooling weights on VALIDATION; VoI uses out-of-fold
    class-conditional LLR distributions from TRAIN."""
    def __init__(self, groups, use_rho=True, C=0.5):
        self.groups, self.use_rho, self.C = groups, use_rho, C
    def _L(self, X, rho=None):
        L = np.column_stack([self.cal[s].llr(X) for s in self.groups])
        if self.use_rho and rho is not None:
            L = L * rho
        return L
    def _rho(self, X):
        return np.column_stack([X["rho_" + s].fillna(1.0).values if ("rho_" + s) in X else np.ones(len(X))
                                for s in self.groups])
    def fit(self, Xtr, ytr, Xva, yva, groups_tr=None):
        from sklearn.linear_model import LogisticRegression
        from sklearn.model_selection import GroupKFold
        self.cal = {s: SourceCalibrator(f, self.C).fit(Xtr, ytr) for s, f in self.groups.items()}
        Lva = self._L(Xva, self._rho(Xva))
        self.stack = LogisticRegression(C=1.0, max_iter=2000).fit(Lva, yva)
        self.w = self.stack.coef_.ravel(); self.b0 = float(self.stack.intercept_[0])
        # out-of-fold LLR samples (train) for VoI predictive distributions
        oof = np.zeros((len(Xtr), len(self.groups)))
        g = groups_tr if groups_tr is not None else np.arange(len(Xtr)) % 5
        for tr, te in GroupKFold(n_splits=5).split(Xtr, ytr, g):
            for j, (s, f) in enumerate(self.groups.items()):
                c = SourceCalibrator(f, self.C).fit(Xtr.iloc[tr], ytr[tr])
                oof[te, j] = c.llr(Xtr.iloc[te])
        oof = oof * (self._rho(Xtr) if self.use_rho else 1.0)
        rng = np.random.default_rng(0)
        self.samples = {}
        for j, s in enumerate(self.groups):
            for yv in (0, 1):
                v = oof[ytr == yv, j]; v = v[v != 0] if (v != 0).any() else v
                self.samples[(s, yv)] = rng.choice(v, size=min(400, len(v)), replace=len(v) < 400) if len(v) else np.zeros(1)
        return self
    def contributions(self, X):
        return self._L(X, self._rho(X)) * self.w
    def predict_logit(self, X, mask=None):
        C = self.contributions(X)
        if mask is not None: C = C * mask
        return self.b0 + C.sum(1)
    def predict(self, X, mask=None):
        return _sigmoid(self.predict_logit(X, mask))

def bayes_risk(p, tau):
    return np.minimum(tau * (1 - p), (1 - tau) * p)

def voi(z, j_name, fusion, tau, j_idx):
    """Expected reduction in Bayes decision risk if evidence source j is acquired."""
    p = _sigmoid(z)
    l1 = fusion.samples[(j_name, 1)]; l0 = fusion.samples[(j_name, 0)]
    w = fusion.w[j_idx]
    e1 = bayes_risk(_sigmoid(z + w * l1), tau).mean()
    e0 = bayes_risk(_sigmoid(z + w * l0), tau).mean()
    return float(bayes_risk(p, tau) - (p * e1 + (1 - p) * e0))

def entropy_gain(z, j_name, fusion, j_idx):
    def H(p): p = np.clip(p, 1e-9, 1 - 1e-9); return -(p * np.log(p) + (1 - p) * np.log(1 - p))
    p = _sigmoid(z); w = fusion.w[j_idx]
    e1 = H(_sigmoid(z + w * fusion.samples[(j_name, 1)])).mean()
    e0 = H(_sigmoid(z + w * fusion.samples[(j_name, 0)])).mean()
    return float(H(p) - (p * e1 + (1 - p) * e0))

# ---------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------
def metrics(y, p):
    from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
    y = np.asarray(y); p = np.asarray(p)
    if y.sum() == 0 or y.sum() == len(y): return dict(AUPRC=np.nan, AUROC=np.nan, Brier=np.nan)
    return dict(AUPRC=average_precision_score(y, p), AUROC=roc_auc_score(y, p), Brier=brier_score_loss(y, np.clip(p, 0, 1)))

def cluster_bootstrap_diff(y, p_a, p_b, clusters, n_boot=1000, seed=0, metric="AUPRC"):
    """Paired cluster (product-code) bootstrap of metric(p_a) - metric(p_b)."""
    from sklearn.metrics import average_precision_score, roc_auc_score
    f = average_precision_score if metric == "AUPRC" else roc_auc_score
    rng = np.random.default_rng(seed)
    cl = np.asarray(clusters); u = np.unique(cl)
    idx = {c: np.where(cl == c)[0] for c in u}
    diffs = []
    for _ in range(n_boot):
        pick = np.concatenate([idx[c] for c in rng.choice(u, len(u), replace=True)])
        yy = y[pick]
        if yy.sum() == 0 or yy.sum() == len(yy): continue
        diffs.append(f(yy, p_a[pick]) - f(yy, p_b[pick]))
    d = np.array(diffs)
    return dict(diff=float(f(y, p_a) - f(y, p_b)), lo=float(np.percentile(d, 2.5)),
                hi=float(np.percentile(d, 97.5)), p_le0=float((d <= 0).mean()))

def cluster_bootstrap_ci(y, p, clusters, n_boot=1000, seed=0, metric="AUPRC"):
    from sklearn.metrics import average_precision_score, roc_auc_score
    f = average_precision_score if metric == "AUPRC" else roc_auc_score
    rng = np.random.default_rng(seed); cl = np.asarray(clusters); u = np.unique(cl)
    idx = {c: np.where(cl == c)[0] for c in u}; vals = []
    for _ in range(n_boot):
        pick = np.concatenate([idx[c] for c in rng.choice(u, len(u), replace=True)])
        if 0 < y[pick].sum() < len(pick): vals.append(f(y[pick], p[pick]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def lead_time_table(cases, p, tau, events, horizon):
    """For every recall event in the evaluated period, earliest alert month before it."""
    df = cases[["code", "mi"]].copy(); df["p"] = p
    rows = []
    for code, e in events:
        w = df[(df.code == code) & (df.mi >= e - horizon) & (df.mi < e)]
        if len(w) == 0: continue
        hit = w[w.p >= tau]
        rows.append(dict(code=code, event_mi=e, detected=len(hit) > 0,
                         lead=(e - hit.mi.min()) if len(hit) else 0))
    return pd.DataFrame(rows)


# AEGIS-MD · Notebook 1 — Real data acquisition from openFDA (CPU, Internet ON)
**Adaptive Evidence-Gap Investigation of Safety signals for Medical Devices**

Downloads only **real public FDA data** through the openFDA API:
1. Device recalls (`/device/recall.json`) and enforcement reports (`/device/enforcement.json`), joined to get the **recall class** (Class I/II/III). These are the **ground-truth labels** and the corpus of historical **precedents**.
2. Selects a panel of device **product codes** (codes with and without Class I recalls).
3. Monthly MAUDE report **counts** per product code (`count=date_received`).
4. A monthly sample of MAUDE **records** per product code (narratives, manufacturer investigation text, lot/model numbers, problem codes, event type, report source).

**Before running:** Settings → Internet **ON**. Strongly recommended: get a free openFDA API key (https://open.fda.gov/apis/authentication/) and add it as a Kaggle secret named `OPENFDA_API_KEY` (Add-ons → Secrets). Without a key openFDA allows only ~1,000 requests/day.

The record download is **resumable**: it stops after `TIME_BUDGET_MIN` minutes and saves progress. Just run the notebook again (or "Save & Run All" again) until it prints `ALL CODES DONE`.

In [ ]:
import os, sys, json, time, math, random
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd())
from aegis_core import *

OUT = out_dir()
TIME_BUDGET_MIN = 600         # stop record download after this many minutes (resume on next run)
N_POS_CODES = 50              # product codes with >=1 Class I recall in the label window
N_NEG_CODES = 30              # product codes with no Class I recall in the whole period
RECORDS_PER_MONTH = 60        # MAUDE records sampled per code-month
MIN_TOTAL_REPORTS = 600       # code must have >= this many MAUDE reports in DATA_START..DATA_END
MAX_TOTAL_REPORTS = 400000
MOCK = os.environ.get("AEGIS_MOCK") == "1"   # local testing only

API_KEY = os.environ.get("OPENFDA_API_KEY", "")
try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret("OPENFDA_API_KEY") or API_KEY
except Exception as _ex:
    print("secret not available:", type(_ex).__name__, str(_ex)[:150])
print("openFDA API key:", "found" if API_KEY else "NOT found (limited to ~1000 requests/day)")
print(json.dumps(CFG, indent=1))

In [ ]:
import requests
BASE = "https://api.fda.gov"
SLEEP = 0.26 if API_KEY else 0.35
_N_CALLS = 0

def get_json(endpoint, params, retries=5):
    """GET with retries. openFDA returns HTTP 404 when a search has no matches -> empty result."""
    global _N_CALLS
    if MOCK:
        import mock_openfda; _N_CALLS += 1
        return mock_openfda.respond(endpoint, dict(params))
    p = dict(params)
    if API_KEY: p["api_key"] = API_KEY
    for a in range(retries):
        try:
            r = requests.get(BASE + endpoint, params=p, timeout=60)
            _N_CALLS += 1
            time.sleep(SLEEP)
            if r.status_code == 404: return {"results": [], "meta": {"results": {"total": 0}}}
            if r.status_code == 403:
                raise PermissionError(f"openFDA 403 for {endpoint}: {r.text[:150]}")
            if r.status_code == 429:
                time.sleep(20 * (a + 1)); continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            print("  retry", a + 1, endpoint, str(e)[:120]); time.sleep(3 * (a + 1))
    raise RuntimeError(f"openFDA request failed: {endpoint} {params}")

def paged(endpoint, search, limit=1000, max_skip=25000):
    out, skip = [], 0
    while True:
        js = get_json(endpoint, {"search": search, "limit": limit, "skip": skip})
        res = js.get("results", [])
        out += res
        total = js.get("meta", {}).get("results", {}).get("total", 0)
        skip += limit
        if not res or skip >= total: break
        if skip + limit > max_skip + 1000:
            print(f"  WARNING: {search} has {total} hits; capped at {max_skip + 1000}. Narrow the date window."); break
    return out

## 1. Recalls + enforcement reports → recall class (ground truth)

In [ ]:
_cached = find_file("recalls.pkl", required=False)
y0, y1 = ym(CFG["DATA_START"]) // 12 - 2, ym(CFG["DATA_END"]) // 12 + 1   # recalls a bit wider than MAUDE window
recall_rows, enf_rows = [], []
for y in (range(y0, y1 + 1) if not _cached else []):
    got = []
    for fmt in (f"[{y}0101 TO {y}1231]", f"[{y}-01-01 TO {y}-12-31]"):
        got = paged("/device/recall.json", f"event_date_initiated:{fmt}")
        if got: break
    recall_rows += got
    enf = paged("/device/enforcement.json", f"recall_initiation_date:[{y}0101 TO {y}1231]")
    enf_rows += enf
    print(y, "recalls:", len(got), " enforcement:", len(enf))

def first(x):
    if isinstance(x, list): return x[0] if x else None
    return x

if _cached:
    rec = pd.read_pickle(_cached); print("loaded cached recalls:", _cached)
else:
  rec = pd.DataFrame([dict(
      event_id=str(r.get("res_event_number") or ""),
      recall_number=str(r.get("product_res_number") or "").strip(),
      code=str(first(r.get("product_code")) or "").strip().upper(),
      firm=r.get("recalling_firm"),
      init_mi=mi_from_yyyymmdd(r.get("event_date_initiated")),
      posted_mi=mi_from_yyyymmdd(r.get("event_date_posted")),
      reason=(r.get("reason_for_recall") or "")[:1500],
      root_cause=r.get("root_cause_description") or "",
      device_name=first((r.get("openfda") or {}).get("device_name")) or (r.get("product_description") or "")[:200],
      klass_rec=str(r.get("classification") or r.get("recall_class") or ""),
  ) for r in recall_rows])
  enf = pd.DataFrame([dict(recall_number=str(e.get("recall_number") or "").strip(),
                           klass_enf=str(e.get("classification") or "")) for e in enf_rows]).drop_duplicates("recall_number")
  rec = rec.merge(enf, on="recall_number", how="left")
  rec["klass"] = np.where(rec.klass_rec.str.startswith("Class"), rec.klass_rec, rec.klass_enf.fillna(""))
  rec = rec[(rec.code != "") & rec.init_mi.notna()].copy()
  rec["posted_mi"] = rec.posted_mi.fillna(rec.init_mi + 2)      # availability time of the recall text
  rec = rec.drop_duplicates(["event_id", "recall_number", "code"]).reset_index(drop=True)
  rec["rec_id"] = np.arange(len(rec))
  print("recall rows:", len(rec), "| class coverage:", (rec.klass.str.startswith("Class")).mean().round(3))
  print(rec.klass.value_counts().head())
rec.to_pickle(f"{OUT}/recalls.pkl")

## 2. Select product-code panel

In [ ]:
d0 = f"{CFG['DATA_START'].replace('-', '')}01"; d1 = f"{CFG['DATA_END'].replace('-', '')}31"
_cc = find_file("codes.pkl", required=False)
COUNT_CACHE = {}
def code_daily_counts(c):
    if c not in COUNT_CACHE:
        js = get_json("/device/event.json", {
            "search": f'device.device_report_product_code:"{c}" AND date_received:[{d0} TO {d1}]', "count": "date_received"})
        COUNT_CACHE[c] = js.get("results", [])
    return COUNT_CACHE[c]
vol = pd.DataFrame()
if not _cc:
    try:
        js = get_json("/device/event.json", {"search": f"date_received:[{d0} TO {d1}]",
                                             "count": "device.device_report_product_code.exact", "limit": 1000})
        vol = pd.DataFrame(js.get("results", [])).rename(columns={"term": "code", "count": "n_maude"})
        vol["code"] = vol.code.str.upper()
        print("codes with MAUDE volume:", len(vol))
    except PermissionError as ex:
        # openFDA requires an API key for cross-code aggregate counts. Fallback: candidate codes from the recall
        # database (positives: Class I recall in the label window; negatives: recalled devices that never had a
        # Class I recall), volume from per-code count queries (allowed without a key).
        print("aggregate count needs a key -> recall-database candidate pool.", str(ex)[:100])
        lab_lo0, lab_hi0 = ym(CFG["PANEL_START"]), ym(CFG["PANEL_END"]) + CFG["HORIZON"]
        isl = rec.klass.isin(CFG["LABEL_CLASSES"])
        pos_pool = sorted(set(rec[isl & rec.init_mi.between(lab_lo0, lab_hi0)].code))
        neg_pool = sorted(set(rec[rec.init_mi.between(ym(CFG["DATA_START"]), ym(CFG["DATA_END"]))].code) - set(rec[isl].code))
        r0 = np.random.default_rng(CFG["SEED"])
        pool = list(r0.permutation(pos_pool)[:N_POS_CODES * 3]) + list(r0.permutation(neg_pool)[:N_NEG_CODES * 4])
        print("candidate pool:", len(pos_pool), "pos /", len(neg_pool), "neg; querying", len(pool))
        vol = pd.DataFrame([(c, sum(r["count"] for r in code_daily_counts(c))) for c in pool], columns=["code", "n_maude"])
if _cc:
    codes = pd.read_pickle(_cc); print("loaded cached code panel:", _cc)
lab_lo, lab_hi = ym(CFG["PANEL_START"]), ym(CFG["PANEL_END"]) + CFG["HORIZON"]
is_lab = rec.klass.isin(CFG["LABEL_CLASSES"])
pos_codes = set(rec[is_lab & rec.init_mi.between(lab_lo, lab_hi)].code)
ever_lab = set(rec[is_lab].code)
if not _cc:
  cand = vol[(vol.n_maude >= MIN_TOTAL_REPORTS) & (vol.n_maude <= MAX_TOTAL_REPORTS)].copy()
  rng = np.random.default_rng(CFG["SEED"])
  pos = cand[cand.code.isin(pos_codes)]
  neg = cand[~cand.code.isin(ever_lab)]
  pos = pos.sample(min(N_POS_CODES, len(pos)), random_state=CFG["SEED"])
  neg = neg.sample(min(N_NEG_CODES, len(neg)), random_state=CFG["SEED"])
  codes = pd.concat([pos.assign(group="pos"), neg.assign(group="neg")]).reset_index(drop=True)
  names = rec.groupby("code").device_name.agg(lambda s: s.dropna().iloc[0] if s.notna().any() else "")
  codes["device_name"] = codes.code.map(names).fillna("")
  print(codes.group.value_counts())
codes.to_pickle(f"{OUT}/codes.pkl")

## 3. Monthly MAUDE counts per product code (full counts, not a sample)

In [ ]:
_ck = find_file("counts.pkl", required=False)
cnt_rows = []
for i, c in enumerate(codes.code if not _ck else []):
    # openFDA returns every daily bucket for date counts, so one call per code covers the whole period
    for r in code_daily_counts(c):
        cnt_rows.append((c, mi_from_yyyymmdd(r["time"]), r["count"]))
    if i % 10 == 0: print(i, c, "calls so far", _N_CALLS)
counts = pd.read_pickle(_ck) if _ck else pd.DataFrame(cnt_rows, columns=["code", "mi", "n"]).groupby(["code", "mi"], as_index=False).n.sum()
counts.to_pickle(f"{OUT}/counts.pkl")
# fill device names that recalls could not provide (negative codes) from the classification endpoint
for i in codes.index[codes.device_name == ""]:
    js = get_json("/device/classification.json", {"search": f'product_code:"{codes.at[i, "code"]}"', "limit": 1})
    if js.get("results"): codes.at[i, "device_name"] = js["results"][0].get("device_name", "")
codes.to_pickle(f"{OUT}/codes.pkl")
print("count rows:", len(counts))

## 4. Monthly sample of MAUDE records (resumable)

In [ ]:
def _j(x):
    if isinstance(x, list): return ";".join(str(v) for v in x if v is not None)
    return "" if x is None else str(x)

def parse_event(e, code):
    dev = (e.get("device") or [{}])
    d = next((x for x in dev if str(x.get("device_report_product_code", "")).upper() == code), dev[0] if dev else {})
    txt_e, txt_m = [], []
    for t in e.get("mdr_text") or []:
        tt = str(t.get("text_type_code", "")).lower()
        (txt_m if "manufacturer" in tt else txt_e).append(str(t.get("text") or ""))
    te, n1 = scrub_recall_mentions(" ".join(txt_e)[:3000])
    tm, n2 = scrub_recall_mentions(" ".join(txt_m)[:3000])
    probs = e.get("product_problems") or []
    tor = e.get("type_of_report") or []
    return dict(mdr_key=str(e.get("mdr_report_key", "")), code=code,
                mi=mi_from_yyyymmdd(e.get("date_received")),
                event_type=str(e.get("event_type", "")), source=str(e.get("report_source_code", "")),
                type_of_report=_j(tor),
                mfr=str(d.get("manufacturer_d_name", "") or ""), brand=str(d.get("brand_name", "") or ""),
                model=str(d.get("model_number", "") or "").strip().upper(),
                lot=str(d.get("lot_number", "") or "").strip().upper(),
                evaluated=str(d.get("device_evaluated_by_manufacturer", "") or ""),
                problems=_j(probs),
                text_event=te[:1200], text_mfr=tm[:1200], scrubbed=n1 + n2)

state_path = f"{OUT}/records_progress.json"
state = json.load(open(state_path)) if os.path.exists(state_path) else {"done": []}
# allow resuming from a previous version of this notebook attached as input
prev = find_file("records_progress.json", required=False)
if prev and prev != state_path and not state["done"]:
    state = json.load(open(prev))
    for f in glob.glob(os.path.join(os.path.dirname(prev), "records_*.pkl")):
        if not os.path.exists(os.path.join(OUT, os.path.basename(f))):
            pd.read_pickle(f).to_pickle(os.path.join(OUT, os.path.basename(f)))
    print("resumed from attached input:", len(state["done"]), "codes done")

# Without an API key the records come from openFDA's bulk quarterly files (no rate limit). Within each
# product-code-month a uniform random sample is kept by bottom-k hashing of the report key.
_have_records = find_file("records.pkl", required=False) is not None   # data notebook already attached
USE_BULK = (not API_KEY) and not MOCK and not _have_records
code_set = set(codes.code)
if USE_BULK:
    import io, zipfile, hashlib, re as _re
    from concurrent.futures import ThreadPoolExecutor
    meta = requests.get(BASE + "/download.json", timeout=120).json()
    parts_all = meta["results"]["device"]["event"]["partitions"]
    def part_q(p):
        m = _re.match(r"(\d{4}) Q(\d)", p.get("display_name", ""))
        return int(m.group(1)) * 12 + (int(m.group(2)) - 1) * 3 if m else None
    lo_, hi_ = ym(CFG["DATA_START"]), ym(CFG["DATA_END"])
    sel = [p for p in parts_all if part_q(p) is not None and lo_ - 2 <= part_q(p) <= hi_]
    print("bulk partitions:", len(sel), "| total MB:", round(sum(float(p["size_mb"]) for p in sel)))
    state.setdefault("bulk_done", [])
    def hkey(k): return int(hashlib.md5(k.encode()).hexdigest()[:12], 16)
    def fetch(url):
        for a in range(6):
            try:
                r = requests.get(url, timeout=600); r.raise_for_status(); return r.content
            except Exception as ex_:
                print("  download retry", a + 1, str(ex_)[:100]); time.sleep(15 * (a + 1))
        raise RuntimeError(url)
    todo = [p for p in sel if p["file"] not in state["bulk_done"]]
    t_start = time.time()
    with ThreadPoolExecutor(2) as pool:
        futs = {i: pool.submit(fetch, todo[i]["file"]) for i in range(min(2, len(todo)))}
        for i, p in enumerate(todo):
            if (time.time() - t_start) / 60 > TIME_BUDGET_MIN:
                print("Time budget reached - run this cell again to continue."); break
            blob = futs.pop(i).result()
            if i + 2 < len(todo): futs[i + 2] = pool.submit(fetch, todo[i + 2]["file"])
            with zipfile.ZipFile(io.BytesIO(blob)) as z:
                js = json.loads(z.read(z.namelist()[0]))
            del blob
            keep = {}; n_bad = 0
            for e in js.get("results", []):
                pcs = {str(d.get("device_report_product_code", "")).upper() for d in (e.get("device") or [])}
                for c in pcs & code_set:
                    try:
                        r = parse_event(e, c)
                    except Exception as ex_:
                        n_bad += 1; continue
                    if r["mi"] != r["mi"]: continue
                    keep.setdefault((c, int(r["mi"])), []).append((hkey(r["mdr_key"] + c), r))
            del js
            rows = []
            for g, lst in keep.items():
                lst.sort(key=lambda x: x[0]); rows += [dict(r, h=h) for h, r in lst[:RECORDS_PER_MONTH]]
            pd.DataFrame(rows).to_pickle(f"{OUT}/bulk_{len(state['bulk_done']):04d}.pkl")
            state["bulk_done"].append(p["file"]); json.dump(state, open(state_path, "w"))
            print(f"{p['display_name']}: kept {len(rows)} (skipped {n_bad} malformed) | {len(state['bulk_done'])}/{len(sel)} files | {(time.time() - t_start) / 60:.1f} min", flush=True)
        for f_ in futs.values(): f_.cancel()
    if len(state["bulk_done"]) >= len(sel):
        Bk = pd.concat([pd.read_pickle(f) for f in sorted(glob.glob(f"{OUT}/bulk_*.pkl"))], ignore_index=True)
        Bk = Bk.sort_values("h").groupby(["code", "mi"], group_keys=False).head(RECORDS_PER_MONTH)
        for c, g in Bk.groupby("code"): g.drop(columns="h").to_pickle(f"{OUT}/records_{c}.pkl")
        state["done"] = list(codes.code); json.dump(state, open(state_path, "w"))
        for f in glob.glob(f"{OUT}/bulk_*.pkl"): os.remove(f)
        del Bk

cmap = counts.set_index(["code", "mi"]).n.to_dict()
t_start = time.time()
months = range(ym(CFG["DATA_START"]), ym(CFG["DATA_END"]) + 1)
for c in (codes.code if not (USE_BULK or _have_records) else []):
    if c in state["done"]: continue
    if (time.time() - t_start) / 60 > TIME_BUDGET_MIN:
        print(f"Time budget reached. {len(state['done'])}/{len(codes)} codes done. RUN THE NOTEBOOK AGAIN to continue."); break
    rows = []
    for m in months:
        if cmap.get((c, m), 0) == 0: continue
        y, mo = m // 12, m % 12 + 1
        last = pd.Period(f"{y}-{mo:02d}").days_in_month
        js = get_json("/device/event.json", {
            "search": f'device.device_report_product_code:"{c}" AND date_received:[{y}{mo:02d}01 TO {y}{mo:02d}{last}]',
            "limit": RECORDS_PER_MONTH})
        rows += [parse_event(e, c) for e in js.get("results", [])]
    pd.DataFrame(rows).to_pickle(f"{OUT}/records_{c}.pkl")
    state["done"].append(c); json.dump(state, open(state_path, "w"))
    print(f"{c}: {len(rows)} records | {len(state['done'])}/{len(codes)} | {(time.time() - t_start) / 60:.1f} min | calls {_N_CALLS}")

In [ ]:
if _have_records:
    print("records.pkl found in attached input - skipping download")
elif len(state["done"]) == len(codes):
    parts = [pd.read_pickle(f"{OUT}/records_{c}.pkl") for c in codes.code if os.path.exists(f"{OUT}/records_{c}.pkl")]
    R = pd.concat(parts, ignore_index=True)
    R = R[R.mi.notna()].drop_duplicates(["mdr_key", "code"]).reset_index(drop=True)
    R["mi"] = R.mi.astype(int); R["rid"] = np.arange(len(R))
    R.to_pickle(f"{OUT}/records.pkl")
    for c in codes.code:
        p = f"{OUT}/records_{c}.pkl"
        if os.path.exists(p): os.remove(p)
    print("ALL CODES DONE. records:", len(R), "| with manufacturer narrative:", (R.text_mfr.str.len() > 20).mean().round(3),
          "| sentences scrubbed for recall mentions:", int(R.scrubbed.sum()))
    summary = dict(n_codes=len(codes), n_records=len(R), n_recalls=len(rec), n_count_rows=len(counts),
                   label_classes=CFG["LABEL_CLASSES"], api_calls=_N_CALLS)
    json.dump(summary, open(f"{OUT}/data_summary.json", "w"), indent=1); print(summary)

In [ ]:
import gc; gc.collect(); print('section done')
